# Building a RAG System based on connection and retrival through mysql connection

### 1. setting up connection with MySql

In [1]:
!pip install mysql-connector-python


   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ------ --------------------------------- 2.6/16.4 MB 25.1 MB/s eta 0:00:01
   ---------- ----------------------------- 4.2/16.4 MB 19.4 MB/s eta 0:00:01
   ---------------- ----------------------- 6.6/16.4 MB 11.8 MB/s eta 0:00:01
   --------------------- ------------------ 8.9/16.4 MB 11.5 MB/s eta 0:00:01
   --------------------------- ------------ 11.3/16.4 MB 12.8 MB/s eta 0:00:01
   ------------------------------- -------- 12.8/16.4 MB 10.7 MB/s eta 0:00:01
   ---------------------------------- ----- 14.2/16.4 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 11.0 MB/s eta 0:00:00


In [2]:
import mysql.connector
connection = mysql.connector.connect(
    host='localhost',
    user='root',
    password=password,
    database='employees'
)
cursor = connection.cursor(buffered=True)
# cursor.execute('SHOW TABLES')
# tables = cursor.fetchall()
# tables
#database
select_query = 'SELECT * FROM employees LIMIT 10'
cursor.execute(select_query)
# 
row = cursor.fetchall()


In [3]:
print(row)

[(10001, datetime.date(1953, 9, 2), 'Georgi', 'Facello', 'M', datetime.date(1986, 6, 26)), (10002, datetime.date(1964, 6, 2), 'Bezalel', 'Simmel', 'F', datetime.date(1985, 11, 21)), (10003, datetime.date(1959, 12, 3), 'Parto', 'Bamford', 'M', datetime.date(1986, 8, 28)), (10004, datetime.date(1954, 5, 1), 'Chirstian', 'Koblick', 'M', datetime.date(1986, 12, 1)), (10005, datetime.date(1955, 1, 21), 'Kyoichi', 'Maliniak', 'M', datetime.date(1989, 9, 12)), (10006, datetime.date(1953, 4, 20), 'Anneke', 'Preusig', 'F', datetime.date(1989, 6, 2)), (10007, datetime.date(1957, 5, 23), 'Tzvetan', 'Zielinski', 'F', datetime.date(1989, 2, 10)), (10008, datetime.date(1958, 2, 19), 'Saniya', 'Kalloufi', 'M', datetime.date(1994, 9, 15)), (10009, datetime.date(1952, 4, 19), 'Sumant', 'Peac', 'F', datetime.date(1985, 2, 18)), (10010, datetime.date(1963, 6, 1), 'Duangkaew', 'Piveteau', 'F', datetime.date(1989, 8, 24))]


In [4]:
row[0]

(10001,
 datetime.date(1953, 9, 2),
 'Georgi',
 'Facello',
 'M',
 datetime.date(1986, 6, 26))

### 2. Instantiating model

In [5]:

import os


#model instantiate 
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://api.together.xyz/v1",
    api_key=os.environ["TOGETHERAI_API_KEY"],
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
)

### Converting query to Code


In [6]:
question = 'How many employees are having salary greater than 90000 in salaries table in salary column . '


In [7]:
prompt = f'''follow the instructions strictly.
only provide the SQL code for the question asked below.
{question}
never provide anything other than that.
no english , no puncutaltuins.'''

In [8]:
prompt = prompt.format({'question':question})
prompt

'follow the instructions strictly.\nonly provide the SQL code for the question asked below.\nHow many employees are having salary greater than 90000 in salaries table in salary column . \nnever provide anything other than that.\nno english , no puncutaltuins.'

In [9]:
query = llm.invoke(prompt)

query = query.content
query

'SELECT COUNT(*) FROM salaries WHERE salary > 90000'

### Executing Query

In [10]:
cursor.execute(query)
result=cursor.fetchall()
result = str(result[0]) #casting in string
result

'(31822,)'

#### Genrate answer

In [13]:
prompt = f''' given the result received {result} of the sql query.
provide the result to the user in max 20 words.
include ``hello your result is``  before result
always say ``Have a great day in the end`` in the end .
'''
final = llm.invoke(prompt)


In [12]:

final.content

'Hello your result is 31822. Have a great day'